# Làm sạch và kiểm tra dữ liệu xe đã qua sử dụng tại UAE

Notebook này giới thiệu bộ dữ liệu, giải thích từng cột và hướng dẫn kiểm tra chất lượng dữ liệu bằng Python cơ bản. Mục tiêu là hiểu dữ liệu trước khi đặt câu hỏi phân tích hoặc xây dựng mô hình dự đoán giá.

**Nguồn:** [UAE Used Car Prices & Features (10K+ Listings) trên Kaggle](https://www.kaggle.com/datasets/alikalwar/uae-used-car-prices-and-features-10k-listings) — Ali Kalwar. **Giấy phép:** CC BY-SA 4.0. Khi chia sẻ lại dữ liệu hoặc sản phẩm phái sinh, hãy ghi công nguồn và tuân thủ điều khoản chia sẻ tương tự của giấy phép.

> `Price` là giá được ghi trong tin đăng. Không có bằng chứng trong các cột rằng đây là giá giao dịch cuối cùng.


## 1. Câu hỏi của dự án

Một câu hỏi phù hợp cho bài phân tích là: **Giá niêm yết xe cũ trong bộ dữ liệu thay đổi như thế nào theo năm sản xuất, số km đã đi, hãng xe, loại thân xe và địa điểm?**

Có thể bắt đầu bằng thống kê mô tả và biểu đồ, sau đó thử một mô hình hồi quy đơn giản để ước tính giá. Đây là mối liên hệ trong dữ liệu quan sát; không tự nó chứng minh một yếu tố gây ra giá cao hay thấp.


## 2. Nạp tệp CSV

Chạy notebook từ thư mục gốc của project; dữ liệu nguồn nằm trong `data/raw/`. Phần này chỉ dùng thư viện chuẩn có sẵn trong Python, không yêu cầu cài thêm gói.


In [4]:
import csv
from pathlib import Path

DATA_FILE = Path("data/raw/uae_used_cars_10k.csv")
if not DATA_FILE.exists():
    raise FileNotFoundError(f"Không tìm thấy {DATA_FILE}. Hãy kiểm tra đường dẫn `data/raw/uae_used_cars_10k.csv`.")

with DATA_FILE.open("r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    columns = reader.fieldnames
    rows = list(reader)

print(f"Số dòng: {len(rows):,}")
print(f"Số cột: {len(columns)}")
print("Tên cột:", columns)


Số dòng: 10,000
Số cột: 12
Tên cột: ['Make', 'Model', 'Year', 'Price', 'Mileage', 'Body Type', 'Cylinders', 'Transmission', 'Fuel Type', 'Color', 'Location', 'Description']


In [5]:
# Xem một vài bản ghi để hiểu cấu trúc hàng dữ liệu
for row in rows[:3]:
    print(row)


{'Make': 'toyota', 'Model': 'camry', 'Year': '2016', 'Price': '47819', 'Mileage': '156500', 'Body Type': 'Sedan', 'Cylinders': '4', 'Transmission': 'Automatic Transmission', 'Fuel Type': 'Gasoline', 'Color': 'Black', 'Location': ' Dubai', 'Description': '2016 toyota camry with Rear camera, Leather seats. Condition: Engine repaired.'}
{'Make': 'kia', 'Model': 'sorento', 'Year': '2013', 'Price': '61250', 'Mileage': '169543', 'Body Type': 'SUV', 'Cylinders': '4', 'Transmission': 'Automatic Transmission', 'Fuel Type': 'Gasoline', 'Color': 'Grey', 'Location': ' Abu Dhabi', 'Description': '2013 kia sorento with Sunroof, Adaptive cruise control, Rear camera, Bluetooth. Condition: Accident history.'}
{'Make': 'mini', 'Model': 'cooper', 'Year': '2023', 'Price': '31861', 'Mileage': '221583', 'Body Type': 'Soft Top Convertible', 'Cylinders': '4', 'Transmission': 'Automatic Transmission', 'Fuel Type': 'Gasoline', 'Color': 'Grey', 'Location': ' Dubai', 'Description': '2023 mini cooper with Adaptive

Mỗi dòng đại diện cho một tin đăng xe trong tệp. Các giá trị được đọc ban đầu dưới dạng chuỗi; các cột số cần được chuyển kiểu trước khi tính toán.


## 3. Từ điển dữ liệu: giải thích 12 cột

Tên cột bên dưới giữ nguyên như trong CSV để tiện đối chiếu. Đơn vị tiền và quãng đường được mô tả theo trang dữ liệu Kaggle.

| Cột | Ý nghĩa | Kiểu nên dùng | Lưu ý khi phân tích |
|---|---|---|---|
| `Make` | Hãng/nhãn hiệu xe (ví dụ Toyota, Nissan) | Phân loại | Chuẩn hóa chữ hoa/thường nếu cần nhóm hãng. |
| `Model` | Dòng/mẫu xe (ví dụ Camry, Altima) | Phân loại | Nhiều giá trị; cùng tên mẫu có thể thuộc hãng khác nhau. |
| `Year` | Năm sản xuất | Số nguyên | Có thể tạo tuổi xe; cần chọn mốc năm nhất quán. |
| `Price` | Giá niêm yết, đơn vị dirham UAE (AED) | Số | Biến kết quả thường được phân tích/dự đoán; kiểm tra giá cực lớn. |
| `Mileage` | Quãng đường xe đã chạy, km | Số nguyên | Giá trị lớn thường gắn với xe đã sử dụng nhiều; kiểm tra phân bố. |
| `Body Type` | Kiểu thân xe (SUV, Sedan, Coupe, …) | Phân loại | Có thể so sánh giá giữa các nhóm. |
| `Cylinders` | Số xi-lanh động cơ | Số nguyên hoặc thiếu | Có một số ô trống; không nên tự động thay bằng 0. |
| `Transmission` | Hộp số (Automatic/Manual Transmission) | Phân loại | Nhóm tự động và số sàn. |
| `Fuel Type` | Loại nhiên liệu (Gasoline, Diesel, Hybrid, Electric) | Phân loại | Các nhóm có kích thước rất khác nhau. |
| `Color` | Màu xe | Phân loại | Thông tin mô tả; có thể không đóng góp nhiều cho mô hình. |
| `Location` | Địa điểm đăng tin tại UAE | Phân loại | Có khoảng trắng thừa ở nhiều giá trị; cần `strip()` trước khi nhóm. |
| `Description` | Mô tả tự do của tin đăng, gồm tiện nghi và ghi chú tình trạng | Văn bản | Có thể chứa thông tin như lịch sử tai nạn/sửa chữa; cần xử lý văn bản nếu khai thác. |

**Phân biệt:** `Description` là văn bản tự do. Không nên coi nội dung “Condition” là một cột tình trạng có cấu trúc cho đến khi trích xuất và kiểm tra từng mẫu.


## 4. Kiểm tra kiểu dữ liệu, ô trống và bản ghi trùng

Tệp có 10.000 dòng và 12 cột. Đoạn mã dưới đây tính ô trống theo từng cột, số giá trị phân biệt và số dòng trùng hoàn toàn.


In [9]:
def is_missing(value):
    return value is None or value.strip().lower() in {"", "none", "unknown", "nan", "null"}

print("Ô trống theo cột:")
for column in columns:
    missing = sum(is_missing(row[column]) for row in rows)
    print(f"- {column}: {missing:,} ({missing / len(rows):.1%})")

distinct = {column: len({row[column] for row in rows}) for column in columns}
print("\nSố giá trị phân biệt:")
for column, count in distinct.items():
    print(f"- {column}: {count:,}")

seen = set()
duplicates = 0
for row in rows:
    key = tuple(row[column] for column in columns)
    if key in seen:
        duplicates += 1
    seen.add(key)
print(f"\nSố dòng trùng hoàn toàn: {duplicates:,}")


Ô trống theo cột:
- Make: 0 (0.0%)
- Model: 0 (0.0%)
- Year: 0 (0.0%)
- Price: 0 (0.0%)
- Mileage: 0 (0.0%)
- Body Type: 0 (0.0%)
- Cylinders: 155 (1.6%)
- Transmission: 0 (0.0%)
- Fuel Type: 0 (0.0%)
- Color: 0 (0.0%)
- Location: 0 (0.0%)
- Description: 0 (0.0%)

Số giá trị phân biệt:
- Make: 65
- Model: 488
- Year: 20
- Price: 9,848
- Mileage: 9,834
- Body Type: 13
- Cylinders: 10
- Transmission: 2
- Fuel Type: 4
- Color: 17
- Location: 16
- Description: 9,985

Số dòng trùng hoàn toàn: 0


### Nhận xét từ lần kiểm tra tệp này

- `Cylinders` có **155 giá trị thiếu/không xác định**: 80 ô trống, 25 chuỗi `None`, 50 chuỗi `Unknown` (1,55%). Cần xem cả ba dạng là chưa biết khi phân tích. Các cột còn lại không có ô trống rỗng.
- Không có dòng trùng hoàn toàn theo cả 12 cột trong lần kiểm tra này.
- `Location` có khoảng trắng ở đầu/cuối trong 9.869 dòng; cần chuẩn hóa trước khi thống kê địa điểm.
- `Description` gần như duy nhất theo từng tin đăng, nên số giá trị khác nhau cao là điều dễ hiểu.

Đây là những đặc điểm quan sát được ở bản CSV tải về; nếu tệp được cập nhật, hãy chạy lại các ô để lấy số liệu mới.


## 5. Chuyển các cột số và xem thống kê cơ bản

Ta chuyển `Year`, `Price`, `Mileage`, `Cylinders` sang số. Giá trị thiếu của `Cylinders` được giữ là `None`, không tự điền tùy tiện.


In [12]:
numeric_columns = ["Year", "Price", "Mileage", "Cylinders"]
for row in rows:
    for column in numeric_columns:
        value = row[column].strip() if row[column] is not None else ""
        row[column] = float(value) if value and value.lower() not in {"none", "unknown", "nan", "null"} else None

def describe(values):
    values = sorted(value for value in values if value is not None)
    n = len(values)
    if not n:
        return None
    return {
        "n": n, "min": values[0],
        "median": (values[(n - 1) // 2] + values[n // 2]) / 2,
        "max": values[-1]
    }

for column in numeric_columns:
    print(column, describe([row[column] for row in rows]))


Year {'n': 10000, 'min': 2005.0, 'median': 2014.0, 'max': 2024.0}
Price {'n': 10000, 'min': 7183.0, 'median': 102766.0, 'max': 14686975.0}
Mileage {'n': 10000, 'min': 10006.0, 'median': 154370.5, 'max': 299996.0}
Cylinders {'n': 9845, 'min': 3.0, 'median': 6.0, 'max': 12.0}


Trong tệp hiện tại, `Year` nằm trong khoảng 2005–2024; trung vị `Price` là **102.766 AED**; trung vị `Mileage` là **154.370,5 km**. Giá lớn nhất cao hơn nhiều trung vị, nên dùng trung vị/phân vị bên cạnh trung bình và kiểm tra các xe đó trước khi kết luận có lỗi.


## 6. Phân bố các nhóm phân loại

Hàm sau đếm tần suất sau khi loại bỏ khoảng trắng thừa. So sánh tần suất giúp phát hiện mất cân bằng nhóm: kết quả của nhóm hiếm thường kém ổn định hơn.


In [15]:
from collections import Counter

def top_values(column, limit=10):
    counts = Counter(
        (row[column] or "").strip() or "(trống)"
        for row in rows
    )
    return counts.most_common(limit)

for column in ["Make", "Body Type", "Transmission", "Fuel Type", "Color", "Location"]:
    print(f"\n{column}")
    for value, count in top_values(column):
        print(f"  {value}: {count:,}")



Make
  mercedes-benz: 1,486
  nissan: 925
  toyota: 893
  bmw: 698
  ford: 541
  land-rover: 538
  porsche: 450
  audi: 397
  jeep: 345
  lexus: 339

Body Type
  SUV: 4,607
  Sedan: 2,790
  Coupe: 837
  Pick Up Truck: 432
  Hatchback: 334
  Sports Car: 270
  Van: 143
  Hard Top Convertible: 139
  Crossover: 136
  Soft Top Convertible: 120

Transmission
  Automatic Transmission: 9,626
  Manual Transmission: 374

Fuel Type
  Gasoline: 9,714
  Diesel: 153
  Electric: 110
  Hybrid: 23

Color
  White: 3,355
  Black: 2,126
  Grey: 1,307
  Silver: 866
  Blue: 686
  Red: 635
  Gold: 238
  Brown: 235
  Green: 154
  Other Color: 78

Location
  Dubai: 8,011
  Sharjah: 999
  Abu Dhabi: 732
  Ajman: 174
  Al Ain: 47
  Ras Al Khaimah: 18
  Umm Al Qawain: 10
  Fujeirah: 9


Trong tệp hiện tại, Dubai chiếm phần lớn tin đăng (8.011/10.000), loại thân xe SUV là nhóm nhiều nhất (4.607/10.000), và xăng là nhiên liệu áp đảo (9.714/10.000). Các nhóm nhiên liệu Hybrid/Electric nhỏ, vì vậy cần thận trọng khi so sánh chúng với Gasoline.


## 7. Kiểm tra dữ liệu bất thường và chuẩn hóa địa điểm

Giá trị cực lớn có thể là xe đặc biệt đắt, lỗi nhập liệu hoặc đặc điểm của cách tạo/thu thập dữ liệu. Không xóa chỉ vì chúng xa trung vị; hãy kiểm tra các bản ghi cụ thể và nêu rõ tiêu chí xử lý.


In [18]:
# Chuẩn hóa khoảng trắng ở Location để tạo bản phân tích sạch riêng
for row in rows:
    row["Location_clean"] = row["Location"].strip()

prices = [row["Price"] for row in rows if row["Price"] is not None]
price_median = describe(prices)["median"]
print(f"Trung vị giá: {price_median:,.0f} AED")
print("10 tin có giá cao nhất:")
for row in sorted(rows, key=lambda r: r["Price"] or 0, reverse=True)[:10]:
    print(row["Make"], row["Model"], int(row["Year"]), f'{row["Price"]:,.0f} AED', row["Location_clean"])


Trung vị giá: 102,766 AED
10 tin có giá cao nhất:
mclaren p1 2020 14,686,975 AED Dubai
mclaren p1 2005 12,909,943 AED Dubai
mclaren elva 2013 10,308,926 AED Dubai
mclaren elva 2016 9,862,754 AED Dubai
mclaren senna 2013 5,314,704 AED Dubai
mercedes-benz slr 2008 4,811,366 AED Dubai
ferrari 599 2005 4,728,227 AED Dubai
mercedes-benz slr 2006 4,249,664 AED Dubai
ferrari 599 2024 4,197,282 AED Dubai
ferrari 599 2017 4,002,508 AED Dubai


Giá trị cao là ứng viên để kiểm tra, không phải kết luận tự động rằng dữ liệu sai. So sánh hãng, mẫu, năm và mô tả tin đăng để quyết định có giữ hay không.


## 8. Phân tích giá sơ bộ theo nhóm

Ví dụ sau tính trung vị giá theo loại thân xe. Trung vị ít bị kéo lệch bởi một vài mức giá rất cao so với trung bình. Mỗi nhóm đều cần xem số lượng mẫu kèm theo.


In [21]:
groups = {}
for row in rows:
    groups.setdefault(row["Body Type"], []).append(row["Price"])

summary = []
for body_type, values in groups.items():
    stats = describe(values)
    summary.append((body_type, stats["n"], stats["median"]))

for body_type, count, median_price in sorted(summary, key=lambda item: item[2], reverse=True):
    print(f"{body_type:28s} n={count:4d}  trung vị={median_price:,.0f} AED")


Sports Car                   n= 270  trung vị=248,578 AED
Soft Top Convertible         n= 120  trung vị=224,186 AED
Coupe                        n= 837  trung vị=214,211 AED
Wagon                        n=  44  trung vị=155,982 AED
Hard Top Convertible         n= 139  trung vị=140,062 AED
SUV                          n=4607  trung vị=118,414 AED
Pick Up Truck                n= 432  trung vị=113,958 AED
Utility Truck                n=  34  trung vị=109,185 AED
Crossover                    n= 136  trung vị=86,282 AED
Other                        n= 114  trung vị=82,957 AED
Sedan                        n=2790  trung vị=70,288 AED
Van                          n= 143  trung vị=51,966 AED
Hatchback                    n= 334  trung vị=50,980 AED


Bảng này mô tả các tin đăng trong mẫu. Khác biệt giá giữa nhóm có thể đồng thời liên quan đến hãng, năm, số km, trang bị hoặc tình trạng; không diễn giải đây là tác động riêng của `Body Type`.


## 9. Giới hạn và hướng làm tiếp

1. Đây là dữ liệu tin đăng xe ở UAE; mẫu có thể không đại diện cho mọi xe đã qua sử dụng hoặc mọi giao dịch.
2. Giá niêm yết có thể khác giá bán thực tế.
3. `Description` chứa văn bản tự do; trích xuất tình trạng cần kiểm tra nhiều cách viết và tránh gán nhãn sai.
4. Nhóm dữ liệu không cân bằng (địa điểm, nhiên liệu, hộp số), nên báo cáo số lượng mẫu và tránh kết luận mạnh từ nhóm quá nhỏ.
5. Nếu xây dựng mô hình, chia tập huấn luyện/kiểm tra, dùng các chỉ số phù hợp, so sánh với mốc dự đoán trung vị, và không đưa thông tin rò rỉ từ mục tiêu vào đặc trưng.

### Tóm tắt

CSV có **10.000 tin đăng, 12 cột**, gồm thuộc tính xe, giá, địa điểm và mô tả. Bước làm sạch quan trọng đầu tiên là xử lý 155 giá trị thiếu/không xác định ở `Cylinders` và loại bỏ khoảng trắng thừa trong `Location`. Câu hỏi dự án khả thi là khảo sát mối liên hệ giữa các thuộc tính xe và **giá niêm yết**.
